# XGboost prediction model

In [12]:
# Import necessary libraries and data
import pandas as pd
import re
import numpy as np
import os
%run 2.3_xgboost_dataloader_companies.py


Loaded: data\final_panel.csv

  final_panel
  Rows: 27,965    Columns: 21

|   # | Column                            | Dtype   |   Non-Null |   Null |
|-----|-----------------------------------|---------|------------|--------|
|   0 | Company name Latin alphabet       | str     |     27,965 |      0 |
|   1 | Year of incorporation             | int64   |     27,965 |      0 |
|   2 | Company Age                       | int64   |     27,965 |      0 |
|   3 | Bankrupt                          | int64   |     27,965 |      0 |
|   4 | NACE Section                      | str     |     27,965 |      0 |
|   5 | NACE Rev. 2, core code (4 digits) | int64   |     27,965 |      0 |
|   6 | Year                              | int64   |     27,965 |      0 |
|   7 | Current assets                    | float64 |     27,941 |     24 |
|   8 | Current liabilities               | float64 |     27,922 |     43 |
|   9 | P/L before tax                    | float64 |     27,930 |     35 |
|  10 | Share

In [14]:
# ============================================================
# CHECK BANKRUPTCY LABEL BEHAVIOUR ACROSS PANEL YEARS
# ============================================================

# How does Bankrupt label vary within companies across years?
bankrupt_by_year = (
    df_final_panel[df_final_panel["Bankrupt"] == 1]
    .groupby("Company name Latin alphabet")["Year"]
    .agg(
        years_observed="count",
        min_year="min",
        max_year="max",
        years_list=list
    )
    .reset_index()
)

print(f"Bankrupt companies in panel: {len(bankrupt_by_year):,}")
print(f"\nYears observed per bankrupt company:")
print(bankrupt_by_year["years_observed"].value_counts().sort_index())

print(f"\nEarliest observation year distribution:")
print(bankrupt_by_year["min_year"].value_counts().sort_index())

# Check if any company switches label — would indicate year-specific labeling
label_switches = (
    df_final_panel.groupby("Company name Latin alphabet")["Bankrupt"]
    .nunique()
)
switchers = label_switches[label_switches > 1]
print(f"\nCompanies with mixed Bankrupt labels (0 and 1): {len(switchers)}")

Bankrupt companies in panel: 1,129

Years observed per bankrupt company:
years_observed
1    362
2    389
3    378
Name: count, dtype: int64

Earliest observation year distribution:
min_year
2022    1034
2023      66
2024      29
Name: count, dtype: int64

Companies with mixed Bankrupt labels (0 and 1): 0


In [15]:
# Estimate filing year as last year with financial data
core_vars = ["Total assets", "Shareholders funds", 
             "Current assets", "Current liabilities", "P/L before tax"]

last_data_year = (
    df_final_panel[df_final_panel["Bankrupt"] == 1]
    .dropna(subset=core_vars, how="all")
    .groupby("Company name Latin alphabet")["Year"]
    .max()
    .rename("estimated_filing_year")
    .reset_index()
)

print(last_data_year["estimated_filing_year"].value_counts().sort_index())

estimated_filing_year
2022    304
2023    372
2024    453
Name: count, dtype: int64


In [16]:
# Merge estimated filing year back to panel
df_final_panel = df_final_panel.merge(
    last_data_year, on="Company name Latin alphabet", how="left"
)

# For active companies filing year is undefined
# For bankrupt companies construct 1-year-ahead label:
#   Row year = estimated_filing_year - 1 → label as bankrupt
#   All other rows → label as 0 (not yet bankrupt at that point)

# Drop bankrupt companies whose last data is 2022
# (no prior year available in our panel)
bankrupt_2022_last = last_data_year[
    last_data_year["estimated_filing_year"] == 2022
]["Company name Latin alphabet"]

df_model = df_final_panel[
    ~df_final_panel["Company name Latin alphabet"].isin(bankrupt_2022_last)
    | (df_final_panel["Bankrupt"] == 0)
].copy()

print(f"Bankrupt companies retained: "
      f"{df_model[df_model['Bankrupt']==1]['Company name Latin alphabet'].nunique():,}")
print(f"Active companies retained:   "
      f"{df_model[df_model['Bankrupt']==0]['Company name Latin alphabet'].nunique():,}")

Bankrupt companies retained: 825
Active companies retained:   8,993


In [19]:
# ── Construct 1-year-ahead bankruptcy label without groupby.apply ─────────────

# Step 1 — build lookup: company → estimated filing year
filing_year_lookup = (
    last_data_year
    .set_index("Company name Latin alphabet")["estimated_filing_year"]
)

# Step 2 — for each row, label = 1 if:
#   company is bankrupt AND row year = filing year - 1
df_model["bankrupt_next_year"] = 0  # default all to 0

# Get filing year for each row via map
df_model["estimated_filing_year"] = (
    df_model.index.map(
        df_model.reset_index()
        .set_index(df_model.index)
        ["Company name Latin alphabet"]
        if "Company name Latin alphabet" in df_model.columns
        else pd.Series(dtype=str)
    )
)

In [20]:
# Check if company name is in index
print(df_model.index.name)
print(df_model.index[:5])
print(df_model.columns.tolist()[:8])

None
RangeIndex(start=0, stop=5, step=1)
['Year of incorporation', 'Company Age', 'Bankrupt', 'NACE Section', 'NACE Rev. 2, core code (4 digits)', 'Year', 'Current assets', 'Current liabilities']
